<a href="https://colab.research.google.com/github/calicartels/understanding-adversarial-attacks-using-flow-matching/blob/main/Research_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Basic imports
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from tqdm.notebook import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Setting up directory and mount drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Making sure we're in /content before creating directories
%cd /content

# 4. Cleaning if directories exist and cloning the original directories
!rm -rf flow_matching
!git clone https://github.com/facebookresearch/flow_matching.git

# 5. Install package
%cd /content/flow_matching
!pip install -e .

# 6. Set up image example directory
%cd /content/flow_matching/examples/image
!pip install -r requirements.txt

# 7. Create output directory and __init__.py files
!mkdir -p output_dir
!touch /content/flow_matching/examples/image/models/__init__.py
!touch /content/flow_matching/examples/image/training/__init__.py

# 8. Copy your saved model from Drive
!cp -r "/content/drive/MyDrive/flow_matching_model/." "/content/flow_matching/examples/image/output_dir/"

# 9. Clean and reset Python path
import sys
sys.path = ['/content/flow_matching/examples/image', '/content/flow_matching'] + sys.path

# 10. Try imports
try:
    # Import model components
    from models.model_configs import instantiate_model
    from training.eval_loop import CFGScaledModel
    print("Model imports successful!")

    # Import flow_matching components
    import flow_matching
    from flow_matching.path import MixtureDiscreteProbPath
    from flow_matching.path.scheduler import PolynomialConvexScheduler
    from flow_matching.solver.ode_solver import ODESolver
    print("All imports successful!")

    # Load model configuration
    from pathlib import Path
    import json

    checkpoint_path = Path("/content/flow_matching/examples/image/output_dir/checkpoint-199.pth")
    args_filepath = checkpoint_path.parent / 'args.json'

    with open(args_filepath, 'r') as f:
        args_dict = json.load(f)

    print("Configuration loaded successfully!")

except Exception as e:
    print(f"Import error: {e}")
    print("\nCurrent working directory:", os.getcwd())
    print("\nPython path:", sys.path)
    print("\nDirectory contents:")
    !ls -R /content/flow_matching/examples/image/models
    !ls -R /content/flow_matching/flow_matching/path

In [ ]:
from argparse import Namespace
import torch.serialization
torch.serialization.add_safe_globals([Namespace])

# 1. Initialize the model
model = instantiate_model(
    architechture=args_dict['dataset'],
    is_discrete='discrete_flow_matching' in args_dict and args_dict['discrete_flow_matching'],
    use_ema=args_dict['use_ema']
)

# 2. Load checkpoint (with weights_only=True to address the warning)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()  # Set to evaluation mode
model.to(device)

# 3. Setup for generation
batch_size = 16  # Number of images to generate
sample_resolution = 32  # CIFAR10 resolution
cfg_weighted_model = CFGScaledModel(model=model)

# 4. Generate images
try:
    x_0 = torch.randn([batch_size, 3, sample_resolution, sample_resolution], dtype=torch.float32, device=device)
    solver = ODESolver(velocity_model=cfg_weighted_model)

    # Get ODE options from args_dict or use defaults
    ode_opts = args_dict.get('ode_options', {})
    step_size = ode_opts.get('step_size', 0.05)  # Default step size if not specified

    synthetic_samples = solver.sample(
        time_grid=torch.tensor([0.0, 1.0], device=device),
        x_init=x_0,
        method=args_dict.get('ode_method', 'euler'),  # Default to euler if not specified
        step_size=step_size,
        atol=ode_opts.get('atol', 1e-5),
        rtol=ode_opts.get('rtol', 1e-5),
        label=torch.tensor(list(range(batch_size)), device=device),
        cfg_scale=args_dict.get('cfg_scale', 1.0)
    )

    # Scale images to [0, 1] range
    synthetic_samples = torch.clamp(synthetic_samples * 0.5 + 0.5, min=0.0, max=1.0)

    # Visualize generated images
    plt.figure(figsize=(20, 20))
    for i in range(batch_size):
        plt.subplot(4, 4, i + 1)
        plt.imshow(synthetic_samples[i].cpu().permute(1, 2, 0).numpy())
        plt.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Generation error: {e}")
    print("\nargs_dict contents:")
    print(json.dumps(args_dict, indent=2))  # Print args_dict for debugging

** the shitty image quality is because of a higer FiD value, close to 5.5.

** Also i trained on only 200 epochs and CiFar needs close to 921 for an ideal train and 3000 for the perfect model.

In [ ]:
# 1. Basic imports
import torch
import torch.nn.functional as F
from pathlib import Path
import json
import matplotlib.pyplot as plt
from argparse import Namespace
import torch.serialization
from models.model_configs import instantiate_model
from training.eval_loop import CFGScaledModel
from flow_matching.solver.ode_solver import ODESolver
import gc

# Clear GPU memory and cache
torch.cuda.empty_cache()
gc.collect()

# Add Namespace to safe globals
torch.serialization.add_safe_globals([Namespace])

# Setup paths and load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = Path("/content/flow_matching/examples/image/output_dir/checkpoint-199.pth")
args_filepath = checkpoint_path.parent / 'args.json'

with open(args_filepath, 'r') as f:
    args_dict = json.load(f)

# Initialize and load model
model = instantiate_model(
    architechture=args_dict['dataset'],
    is_discrete='discrete_flow_matching' in args_dict and args_dict['discrete_flow_matching'],
    use_ema=args_dict['use_ema']
)
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint["model"])
del checkpoint
torch.cuda.empty_cache()

model.eval()
model = model.to(device)

# Modified FGSM attack for Flow Matching model
def fgsm_attack(model, image, label, epsilon=0.03):
    perturbed_image = image.clone().detach().requires_grad_(True)

    # Create timestep tensor (assuming t=0.5 for attack)
    timesteps = torch.ones((image.shape[0],), device=device) * 0.5

    # Forward pass with required arguments
    with torch.enable_grad():
        output = model(perturbed_image, timesteps, extra={'y': label})
        # For Flow Matching models, we'll use the difference between output and input as loss
        loss = F.mse_loss(output, perturbed_image)

        # Backward pass
        loss.backward()

        # Create adversarial example
        adversarial_image = perturbed_image + epsilon * perturbed_image.grad.sign()
        adversarial_image = torch.clamp(adversarial_image, -1, 1)

    return adversarial_image.detach()

# Generate samples and create adversarial examples
try:
    # Setup generation
    batch_size = 8
    sample_resolution = 32
    cfg_weighted_model = CFGScaledModel(model=model)

    # Generate original samples
    x_0 = torch.randn([batch_size, 3, sample_resolution, sample_resolution],
                      dtype=torch.float32, device=device)
    solver = ODESolver(velocity_model=cfg_weighted_model)

    # Get ODE options
    ode_opts = args_dict.get('ode_options', {})
    step_size = ode_opts.get('step_size', 0.05)

    with torch.no_grad():
        synthetic_samples = solver.sample(
            time_grid=torch.tensor([0.0, 1.0], device=device),
            x_init=x_0,
            method=args_dict.get('ode_method', 'heun2'),  # Using heun2 as specified in args
            step_size=step_size,
            atol=ode_opts.get('atol', 1e-5),
            rtol=ode_opts.get('rtol', 1e-5),
            label=torch.tensor(list(range(batch_size)), device=device),
            cfg_scale=args_dict.get('cfg_scale', 0.0)  # Using cfg_scale from args
        )

    # Scale to [0, 1] range
    synthetic_samples = torch.clamp(synthetic_samples * 0.5 + 0.5, min=0.0, max=1.0)

    # Create adversarial examples
    adversarial_samples = []
    for i in range(batch_size):
        image = synthetic_samples[i].unsqueeze(0)
        label = torch.tensor([i], device=device)
        adv_image = fgsm_attack(model, image, label)
        adversarial_samples.append(adv_image)
        torch.cuda.empty_cache()

    adversarial_samples = torch.cat(adversarial_samples)

    # Move to CPU for visualization
    synthetic_samples = synthetic_samples.cpu()
    adversarial_samples = adversarial_samples.cpu()

    # Visualization
    plt.figure(figsize=(20, 10))
    for i in range(batch_size):
        # Original image
        plt.subplot(2, batch_size, i + 1)
        plt.imshow(synthetic_samples[i].permute(1, 2, 0).numpy())
        plt.title('Original')
        plt.axis('off')

        # Adversarial image
        plt.subplot(2, batch_size, i + batch_size + 1)
        plt.imshow(adversarial_samples[i].permute(1, 2, 0).numpy())
        plt.title('Adversarial')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Generation/Attack error: {e}")
    print("\nargs_dict contents:")
    print(json.dumps(args_dict, indent=2))

finally:
    # Clean up
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
def improved_fgsm_attack(model, image, target_class_idx, epsilon=0.1):  # Increased epsilon
    """
    Improved FGSM attack with targeted misclassification
    """
    image = image.clone().detach().requires_grad_(True)

    # Create target tensor
    target = torch.tensor([target_class_idx], device=device)

    # Multiple attack iterations
    for _ in range(5):  # Added multiple iterations
        image.requires_grad_(True)
        output = classifier(image)
        loss = F.cross_entropy(output, target)
        loss.backward()

        # Create perturbation
        perturbation = epsilon * image.grad.data.sign()

        # Update image
        image = torch.clamp(image + perturbation, -1, 1).detach()

    return image

# Try the improved attack
try:
    # Take one sample
    original = synthetic_samples[0].unsqueeze(0).to(device)

    # Get original prediction
    with torch.no_grad():
        orig_output = classifier(original)
        orig_class = orig_output.argmax().item()
        orig_conf = F.softmax(orig_output, dim=1).max().item()

    print(f"\nOriginal Classification:")
    print(f"Class: {orig_class}, Confidence: {orig_conf:.4f}")

    # Try different epsilon values and target classes
    epsilon_values = [0.05, 0.1, 0.15, 0.2]
    # Target classes related to aircraft/vehicles
    target_classes = [
        404,  # airliner
        751,  # wing
        895,  # aircraft carrier
        627,  # helicopter
    ]

    for epsilon in epsilon_values:
        print(f"\nTrying epsilon = {epsilon}")
        for target_class in target_classes:
            print(f"\nTrying to misclassify as class {target_class}")

            # Generate adversarial example
            adv_image = improved_fgsm_attack(model, original, target_class, epsilon=epsilon)

            # Get adversarial prediction
            with torch.no_grad():
                adv_output = classifier(adv_image)
                adv_class = adv_output.argmax().item()
                adv_conf = F.softmax(adv_output, dim=1).max().item()

            print(f"New Classification - Class: {adv_class}, Confidence: {adv_conf:.4f}")

            # Only show visualization if classification changed
            if adv_class != orig_class:
                plt.figure(figsize=(15, 5))

                plt.subplot(1, 3, 1)
                plt.imshow(original[0].cpu().detach().permute(1, 2, 0).numpy())
                plt.title(f'Original\nClass: {orig_class}\nConf: {orig_conf:.4f}')
                plt.axis('off')

                plt.subplot(1, 3, 2)
                plt.imshow(adv_image[0].cpu().detach().permute(1, 2, 0).numpy())
                plt.title(f'Adversarial\nClass: {adv_class}\nConf: {adv_conf:.4f}')
                plt.axis('off')

                plt.subplot(1, 3, 3)
                perturbation = (adv_image - original)[0].cpu().detach().permute(1, 2, 0).numpy()
                plt.imshow(np.abs(perturbation) * 5)
                plt.colorbar()
                plt.title('Perturbation (x5)')
                plt.axis('off')

                plt.tight_layout()
                plt.show()

except Exception as e:
    print(f"Attack error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def pgd_attack(model, image, target_class_idx, epsilon=0.03, alpha=0.01, num_iter=40):
    """
    PGD Attack implementation
    Args:
        model: target model
        image: original image
        target_class_idx: target class for adversarial example
        epsilon: maximum perturbation
        alpha: step size
        num_iter: number of iterations
    """
    # Initialize
    perturbed_image = image.clone().detach()
    # Add small random noise to start
    perturbed_image = perturbed_image + torch.empty_like(image).uniform_(-epsilon/2, epsilon/2)
    perturbed_image = torch.clamp(perturbed_image, -1, 1)

    # Create target tensor
    target = torch.tensor([target_class_idx], device=device)

    for i in range(num_iter):
        perturbed_image.requires_grad = True

        # Forward pass
        output = classifier(perturbed_image)
        loss = F.cross_entropy(output, target)

        # Backward pass
        model.zero_grad()
        loss.backward()

        # Get gradient
        grad = perturbed_image.grad.data

        # Update image - gradient ascent since we want to maximize loss
        adv_image = perturbed_image + alpha * grad.sign()

        # Project back to epsilon ball and valid image space
        eta = torch.clamp(adv_image - image, min=-epsilon, max=epsilon)
        perturbed_image = torch.clamp(image + eta, min=-1, max=1).detach()

    return perturbed_image

# Compare FGSM and PGD attacks
try:
    # Take one sample
    original = synthetic_samples[0].unsqueeze(0).to(device)

    # Get original prediction
    with torch.no_grad():
        orig_output = classifier(original)
        orig_class = orig_output.argmax().item()
        orig_conf = F.softmax(orig_output, dim=1).max().item()

    print(f"\nOriginal Classification:")
    print(f"Class: {orig_class}, Confidence: {orig_conf:.4f}")

    # Test parameters
    epsilons = [0.03, 0.05, 0.1]
    target_classes = [404, 895]  # Using classes that worked well with FGSM

    for epsilon in epsilons:
        print(f"\nTesting epsilon = {epsilon}")
        for target_class in target_classes:
            print(f"\nTarget class: {target_class}")

            # Generate FGSM adversarial example
            fgsm_image = improved_fgsm_attack(model, original, target_class, epsilon=epsilon)

            # Generate PGD adversarial example
            pgd_image = pgd_attack(model, original, target_class, epsilon=epsilon)

            # Get predictions
            with torch.no_grad():
                # FGSM predictions
                fgsm_output = classifier(fgsm_image)
                fgsm_class = fgsm_output.argmax().item()
                fgsm_conf = F.softmax(fgsm_output, dim=1).max().item()

                # PGD predictions
                pgd_output = classifier(pgd_image)
                pgd_class = pgd_output.argmax().item()
                pgd_conf = F.softmax(pgd_output, dim=1).max().item()

            print(f"FGSM - New Class: {fgsm_class}, Confidence: {fgsm_conf:.4f}")
            print(f"PGD  - New Class: {pgd_class}, Confidence: {pgd_conf:.4f}")

            # Visualize results
            plt.figure(figsize=(20, 5))

            # Original
            plt.subplot(1, 4, 1)
            plt.imshow(original[0].cpu().detach().permute(1, 2, 0).numpy())
            plt.title(f'Original\nClass: {orig_class}\nConf: {orig_conf:.4f}')
            plt.axis('off')

            # FGSM
            plt.subplot(1, 4, 2)
            plt.imshow(fgsm_image[0].cpu().detach().permute(1, 2, 0).numpy())
            plt.title(f'FGSM\nClass: {fgsm_class}\nConf: {fgsm_conf:.4f}')
            plt.axis('off')

            # PGD
            plt.subplot(1, 4, 3)
            plt.imshow(pgd_image[0].cpu().detach().permute(1, 2, 0).numpy())
            plt.title(f'PGD\nClass: {pgd_class}\nConf: {pgd_conf:.4f}')
            plt.axis('off')

            # Perturbation comparison
            plt.subplot(1, 4, 4)
            fgsm_pert = torch.norm((fgsm_image - original)[0], dim=0).cpu().detach()
            pgd_pert = torch.norm((pgd_image - original)[0], dim=0).cpu().detach()
            plt.plot(fgsm_pert.mean(dim=1), label='FGSM')
            plt.plot(pgd_pert.mean(dim=1), label='PGD')
            plt.title('Perturbation Magnitude\nper Row')
            plt.legend()

            plt.tight_layout()
            plt.show()

except Exception as e:
    print(f"Attack error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def generate_transformation_path(original_img, adversarial_img, num_steps=10):
    """Generate interpolated images between original and adversarial"""
    alphas = torch.linspace(0, 1, num_steps, device=device)
    path = []

    for alpha in alphas:
        orig = original_img.squeeze(0)
        adv = adversarial_img.squeeze(0)
        interpolated = orig * (1 - alpha) + adv * alpha
        path.append(interpolated)

    return torch.stack(path)

def extract_features(model, image_batch, target_types=(nn.Conv2d, nn.Linear)):
    """Extract features from multiple layers"""
    features = {}
    handles = []

    def hook_fn(name):
        def forward_hook(module, input, output):
            features[name] = output.detach()
        return forward_hook

    # Register hooks for multiple layers
    for name, module in model.named_modules():
        if isinstance(module, target_types):
            if 'input_blocks' in name or 'middle_block' in name:  # Focus on these blocks
                handle = module.register_forward_hook(hook_fn(name))
                handles.append(handle)
                print(f"Registered hook for layer: {name}")

    # Forward pass
    timesteps = torch.ones((image_batch.shape[0],), device=device) * 0.5
    labels = torch.zeros((image_batch.shape[0],), device=device)
    with torch.no_grad():
        _ = model(image_batch, timesteps, extra={'y': labels})

    # Remove hooks
    for handle in handles:
        handle.remove()

    return features

def visualize_feature_flow(path_images, model, num_components=2):
    """Visualize the flow of features in 2D with enhanced visualization"""
    import numpy as np
    from sklearn.decomposition import PCA

    # Extract features for all images in the path
    all_features = []
    feature_maps = {}

    for img in path_images:
        features = extract_features(model, img.unsqueeze(0))
        # Aggregate features from different layers
        combined_features = []
        for name, feat_map in features.items():
            if name not in feature_maps:
                feature_maps[name] = []
            flat_features = feat_map.view(feat_map.size(0), -1).cpu().numpy()
            feature_maps[name].append(flat_features)
            combined_features.append(flat_features)
        all_features.append(np.concatenate(combined_features, axis=1))

    # Stack all features
    all_features = np.vstack(all_features)

    # Apply PCA
    pca = PCA(n_components=num_components)
    features_2d = pca.fit_transform(all_features)

    # Create visualization
    plt.figure(figsize=(20, 10))

    # Plot feature trajectory
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1],
                         c=np.arange(len(features_2d)),
                         cmap='viridis',
                         s=200)
    plt.colorbar(scatter, label='Step')

    # Add arrows with increased visibility
    for i in range(len(features_2d)-1):
        plt.arrow(features_2d[i, 0], features_2d[i, 1],
                 features_2d[i+1, 0] - features_2d[i, 0],
                 features_2d[i+1, 1] - features_2d[i, 1],
                 head_width=0.2, head_length=0.3, fc='red', ec='red',
                 alpha=0.6, width=0.05)

    plt.title('Feature Space Trajectory', fontsize=14)
    plt.xlabel('First Principal Component', fontsize=12)
    plt.ylabel('Second Principal Component', fontsize=12)
    plt.grid(True, alpha=0.3)

    # Plot image sequence
    plt.subplot(1, 2, 2)
    for i, img in enumerate(path_images):
        plt.subplot(3, 4, i+1)
        plt.imshow(img.cpu().permute(1, 2, 0).numpy())
        plt.axis('off')
        plt.title(f'Step {i}')

    plt.tight_layout()
    plt.show()

# First, let's print the model architecture
print("Model Architecture:")
print(model)

# Then try the visualization
try:
    original = synthetic_samples[0].unsqueeze(0).to(device)
    adversarial = adversarial_samples[0].unsqueeze(0).to(device)
    path_images = generate_transformation_path(original, adversarial, num_steps=10)
    visualize_feature_flow(path_images, model)

except Exception as e:
    print(f"Visualization error: {e}")
    import traceback
    traceback.print_exc()